In [1]:
import data_preparation
import models
import hydra 
from hydra import compose, initialize
import torch 
import matplotlib.pyplot as plt 
from tqdm import tqdm
import os
import numpy as np
from matplotlib.patches import Rectangle
from matplotlib.cm import get_cmap
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from sklearn.decomposition import PCA

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [2]:
initialize(version_base=None, config_path="../config")
cfg = compose(config_name="train_general.yaml", overrides=['dataset=climsim_from_npy', 'model=squeezeformer', 'testing=ft'])
print(cfg.dataset)
print(cfg.model)

{'dataset_name': 'climsim_from_npy', 'input_dim': 94, 'output_dim': 98, 'levels': 45, 'normalize': True, 'normalize_targets': True, 'target_group': '${target_group}', 'data_dir': '/work/scratch-pw5/bradlesc/climsim/processed/group_by_months/', 'input_file': 'input.npy', 'target_file': 'target.npy', 'output_scale_file_path': '/home/users/bradlesc/projects/ClimSim/preprocessing/normalizations/outputs/output_scale.nc', 'v1_targets': ['ptend_t', 'ptend_q0001', 'cam_out_NETSW', 'cam_out_FLWDS', 'cam_out_PRECSC', 'cam_out_PRECC', 'cam_out_SOLS', 'cam_out_SOLL', 'cam_out_SOLSD', 'cam_out_SOLLD'], 'split': {'train': 0.8, 'val': 0.1, 'test': 0.1}, 'dataset_testing_sample_rates': {'quick': 10000, 'reduced': 100, 'full': 7}, 'group_method': 'group_by_months', 'group_by_months': {'num_groups': 4, 'target_group': '${target_group}', 'groups': {'DJF': [], 'MAM': [], 'JJA': [], 'SON': []}, 'test_group': {'SON': None}, 'val_group': {'SON': None}}, 'general_dataset_config': {'num_workers': '${testing.nu

# Peformance Analysis 004 

A note book to look at latents ...

In [3]:
def reshape_to_standard_format(x: np.ndarray) -> np.ndarray:
        """
        Reshapes the output to standard format (batch, variables) from (batch, levels, features)
        Args:
            x: (torch.Tensor) (batch, levels, features) output data from the model
        Returns:
            reshaped_x: (torch.Tensor) (batch, features) reshaped output data
        """

        first_two = x[:, :, 0:2].transpose(0, 2, 1).reshape(x.shape[0], -1)
        means_2_to_9 = x[:, :, 2:10].mean(axis=1)

        # concatenate once
        output = np.concatenate([first_two, means_2_to_9], axis=1)
        return output


In [4]:
def unnormalize(data, normalisation_stats):

    return data * (normalisation_stats['target_max'] - normalisation_stats['target_min']) + normalisation_stats['target_mean']

def normalize(data, normalisation_stats):
    return (data - normalisation_stats['target_mean']) / (normalisation_stats['target_max'] - normalisation_stats['target_min'])

In [5]:
# Getting normalisation stats
cfg.dataset.group_by_months.target_group = "MAM"
mam_trainset = data_preparation.ClimSimNpyDataset(
    cfg.dataset,
    cfg.testing.dataset_testing_type,
    "train",
    normalisation_stats=None,
    model=None,
    seed=0,
)

mam_trained_normalisation_stats = mam_trainset.normalisation_stats
mam_trainset_input = mam_trainset.input.numpy()

logging.info("---")

cfg.dataset.group_by_months.target_group = "SON"
son_trainset = data_preparation.ClimSimNpyDataset(
    cfg.dataset,
    cfg.testing.dataset_testing_type,
    "train",
    normalisation_stats=None,
    model=None,
    seed=0,
)

son_trained_normalisation_stats = son_trainset.normalisation_stats
son_trainset_input = son_trainset.input.numpy()

2026-01-06 15:20:55,063 - INFO - Building dataset for group: MAM
2026-01-06 15:20:55,064 - INFO - Using sample rate directory: sample_rate_7
2026-01-06 15:21:02,245 - INFO - Loaded input shape: (2907264, 94)
2026-01-06 15:21:02,245 - INFO - Loaded target shape: (2907264, 98)
2026-01-06 15:21:02,368 - INFO - Using: train with 2325811 samples
2026-01-06 15:21:08,182 - INFO - Calculating normalization statistics from data.
2026-01-06 15:21:11,038 - INFO - Calculating normalization statistics from data for targets.
2026-01-06 15:21:14,799 - INFO - Data is in MLP format; no conversion needed.
2026-01-06 15:21:14,801 - INFO - ---
2026-01-06 15:21:14,802 - INFO - Building dataset for group: SON
2026-01-06 15:21:14,802 - INFO - Using sample rate directory: sample_rate_7
2026-01-06 15:21:21,845 - INFO - Loaded input shape: (2875392, 94)
2026-01-06 15:21:21,846 - INFO - Loaded target shape: (2875392, 98)
2026-01-06 15:21:21,901 - INFO - Using: train with 2300313 samples
2026-01-06 15:21:25,879 -

In [6]:
path_to_data = '/work/scratch-pw5/bradlesc/climsim/temp/p2.1.4.9_analysis/norm_outputs_001/'
!ls {path_to_data}

train_group_MAM  train_group_SON


In [ ]:
son_trained_son_latents = np.load(os.path.join(path_to_data, 'train_group_SON', 'SON_test_latents.npy'))


In [ ]:

# SON trained model

#   SON test set
son_trained_son_input = reshape_to_standard_format(np.load(os.path.join(path_to_data, 'train_group_SON', 'SON_test_inputs.npy')))
son_trained_son_preds = np.load(os.path.join(path_to_data, 'train_group_SON', 'SON_test_outputs.npy'))
son_trained_son_targs = np.load(os.path.join(path_to_data, 'train_group_SON', 'SON_test_targets.npy'))
son_trained_son_latents = np.load(os.path.join(path_to_data, 'train_group_SON', 'SON_test_latents.npy'))
# son_trained_son_abs_error = np.abs(son_trained_son_preds - son_trained_son_targs)

#   MAM test set
son_trained_mam_input = reshape_to_standard_format(np.load(os.path.join(path_to_data, 'train_group_SON', 'MAM_test_inputs.npy')))
son_trained_mam_preds = np.load(os.path.join(path_to_data, 'train_group_SON', 'MAM_test_outputs.npy'))
son_trained_mam_targs = np.load(os.path.join(path_to_data, 'train_group_SON', 'MAM_test_targets.npy'))
son_trained_mam_latents = np.load(os.path.join(path_to_data, 'train_group_SON', 'MAM_test_latents.npy'))
# son_trained_mam_abs_error = np.abs(son_trained_mam_preds - son_trained_mam_targs)


# MAM trained model

#   SON test set
mam_trained_son_input = reshape_to_standard_format(np.load(os.path.join(path_to_data, 'train_group_MAM', 'SON_test_inputs.npy')))
mam_trained_son_preds = np.load(os.path.join(path_to_data, 'train_group_MAM', 'SON_test_outputs.npy'))
mam_trained_son_targs = np.load(os.path.join(path_to_data, 'train_group_MAM', 'SON_test_targets.npy'))
mam_trained_son_latents = np.load(os.path.join(path_to_data, 'train_group_MAM', 'SON_test_latents.npy'))
# mam_trained_son_abs_error = np.abs(mam_trained_son_preds - mam_trained_son_targs)

#   MAM test set
mam_trained_mam_input = reshape_to_standard_format(np.load(os.path.join(path_to_data, 'train_group_MAM', 'MAM_test_inputs.npy')))
mam_trained_mam_preds = np.load(os.path.join(path_to_data, 'train_group_MAM', 'MAM_test_outputs.npy'))
mam_trained_mam_targs = np.load(os.path.join(path_to_data, 'train_group_MAM', 'MAM_test_targets.npy'))
mam_trained_mam_latents = np.load(os.path.join(path_to_data, 'train_group_MAM', 'MAM_test_latents.npy'))
# mam_trained_mam_abs_error = np.abs(mam_trained_mam_preds - mam_trained_mam_targs)


#   JJA test set
# mam_trained_jja_input = reshape_to_standard_format(np.load(os.path.join(path_to_data, 'train_group_MAM', 'JJA_test_inputs.npy')))
# mam_trained_jja_preds = np.load(os.path.join(path_to_data, 'train_group_MAM', 'JJA_test_outputs.npy'))
# mam_trained_jja_targs = np.load(os.path.join(path_to_data, 'train_group_MAM', 'JJA_test_targets.npy'))
# mam_trained_jja_latents = np.load(os.path.join(path_to_data, 'train_group_MAM', 'JJA_test_latents.npy'))
# mam_trained_jja_abs_error = np.abs(mam_trained_jja_preds - mam_trained_jja_targs)

nvar = son_trained_son_preds.shape[1]

In [ ]:
son_trained_son_targs_norm = normalize(son_trained_son_targs, son_trained_normalisation_stats)
mam_trained_son_targs_norm = normalize(mam_trained_son_targs, mam_trained_normalisation_stats)

son_trained_son_preds_unnorm = unnormalize(son_trained_son_preds, son_trained_normalisation_stats)
mam_trained_son_preds_unnorm = unnormalize(mam_trained_son_preds, mam_trained_normalisation_stats)